# Étape 9 — Le vrai graphe LLM de Junyi : précision, rappel et effet sur le KT

Ce notebook relie **RQ1** (qualité du graphe LLM) et **RQ2** (effet sur le knowledge tracing).

- **Partie A** : on prolonge la courbe des suppressions (E4) à **70 % et 90 %**, car le LLM oublie 60 à 75 % des flèches.
- **Partie B** : Qwen2.5-7B évalue les paires d'exercices de Junyi d'un **même thème**. On en tire plusieurs graphes, du plus **prudent** (peu de flèches) au plus **généreux** (beaucoup de flèches), puis on mesure leur précision et leur rappel face au graphe expert et on entraîne PrereqKT avec chacun.

**Hypothèse testée :** comme les fausses flèches coûtent peu et que les oublis coûtent cher, un graphe LLM **généreux** devrait mieux servir le KT qu'un graphe **prudent**.

**GPU T4 obligatoire.** Durée : partie A environ 15 min ; partie B environ 2 à 3 h (génération puis entraînements). Tout reprend automatiquement en cas de déconnexion.

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2, lr=1e-3, batch=64, epoques_max=60, patience=5)
GRAINE = 42

In [ ]:
# Cellule 2 — Séquences, graphe expert, et repérage des PREMIÈRES tentatives
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv')).sort_values(['uid', 'morceau'])
    U, Q, R, F = [], [], [], []
    vus = {}
    for uid, ex, rep in zip(df['uid'], df['exercices'], df['reponses']):
        q = np.array(ex.split(','), dtype=np.int64)
        r = np.array(rep.split(','), dtype=np.int64)
        deja = vus.setdefault(uid, set())          # exercices déjà tentés par cet élève (tous morceaux confondus)
        f = np.zeros(len(q), dtype=bool)
        for i, x in enumerate(q):
            if x not in deja:
                f[i] = True
                deja.add(x)
        U.append(uid); Q.append(q); R.append(r); F.append(f)
    return np.array(U), Q, R, F

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r, f) in donnees.items():
    n_rep = sum(map(len, q)); n_first = sum(int(x.sum()) for x in f)
    print(f'{p:5s} : {len(set(u)):6,} élèves | {n_rep:9,} réponses | premières tentatives : {n_first:8,} ({n_first/n_rep:.1%})')

In [ ]:
# Cellule 3 — Tenseurs (Q exercices, R réponses, M masque, F première tentative)
def tenseurs(q_list, r_list, f_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64); R = np.zeros_like(Q)
    M = np.zeros((len(q_list), L), dtype=bool);     F = np.zeros_like(M)
    for i, (q, r, f) in enumerate(zip(q_list, r_list, f_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)], F[i, :len(f)] = q, r, True, f
    return tuple(torch.from_numpy(x) for x in (Q, R, M, F))

def sous_ensemble_train(fraction, graine=GRAINE):
    u, q, r, f = donnees['train']
    if fraction >= 1:
        return q, r, f
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    gardes = set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx], [f[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('OK')

### Les modèles

- **GraphKT** : identique à l'étape 5 (le graphe lisse les représentations des exercices, avec un canal « prérequis » et un canal « successeurs »).
- **PrereqKT** : GraphKT **plus** 6 indicateurs calculés à chaque instant pour l'exercice à prédire :
  taux de réussite moyen de l'élève sur ses **prérequis** déjà tentés, part des prérequis déjà tentés, présence de prérequis, et les trois mêmes indicateurs pour ses **successeurs**.
  C'est la traduction directe de l'hypothèse « *s'il a échoué sur A, il échouera plus souvent sur B* ».
  Avec un graphe vide, ces indicateurs valent 0 : PrereqKT redevient GraphKT sans graphe.

In [ ]:
# Cellule 4 — Modèles
def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)

class GraphKT(nn.Module):
    def __init__(self, n, hp, A, indicateurs=False):
        super().__init__()
        d, h = hp['dim'], hp['hidden']
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : successeurs de i
        self.n, self.indicateurs = n, indicateurs
        self.E = nn.Embedding(n, d)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, h, batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        n_ind = 6 if indicateurs else 0
        self.pred = nn.Sequential(nn.Linear(h + d + n_ind, h), nn.ReLU(), nn.Linear(h, 1))

    def representations(self):
        H = self.E.weight
        a_pre = (self.P_pre.sum(1, keepdim=True) > 0).float()
        a_suc = (self.P_suc.sum(1, keepdim=True) > 0).float()
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * a_pre + torch.relu(c['suc'](self.P_suc @ H)) * a_suc
        return H

    def calcul_indicateurs(self, q, r):
        # Historique cumulé par exercice jusqu'à l'instant t : nombre d'essais et de réussites (B, L-1, N)
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float()
        taux = reussites / essais.clamp(min=1)
        sortie = []
        for P in (self.P_pre, self.P_suc):
            voisins = (P[q[:, 1:]] > 0).float()                 # voisins de l'exercice à prédire (B, L-1, N)
            n_vois = voisins.sum(-1)
            n_tentes = (voisins * tente).sum(-1)
            moy = (voisins * tente * taux).sum(-1) / n_tentes.clamp(min=1)
            sortie += [moy, n_tentes / n_vois.clamp(min=1), (n_vois > 0).float()]
        return torch.stack(sortie, -1)

    def forward(self, q, r):
        H = self.representations()
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        x = [self.drop(h), H[q[:, 1:]]]
        if self.indicateurs:
            x.append(self.calcul_indicateurs(q, r))
        return self.pred(torch.cat(x, dim=-1)).squeeze(-1)

print('OK')

In [ ]:
# Cellule 5 — Graphes : expert, aucun, aléatoire, et corruption contrôlée E1–E5
import networkx as nx

def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def corrompre(aretes, type_err, taux, graine=GRAINE, topics=exos['topic'].values):
    rng = random.Random(10_000 + graine)
    E = set(map(tuple, aretes))
    k = int(round(taux * len(E)))
    noeuds = list(range(N))
    if type_err == 'E1':      # fausses flèches aléatoires
        while k > 0:
            a, b = rng.sample(noeuds, 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E2':    # fausses flèches plausibles : même thème
        par_topic = {}
        for i, t in enumerate(topics):
            par_topic.setdefault(t, []).append(i)
        groupes = [g for g in par_topic.values() if len(g) >= 2]
        essais = 0
        while k > 0 and essais < 100 * len(E) + 1000:
            essais += 1
            a, b = rng.sample(rng.choice(groupes), 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E3':    # inversions
        for a, b in rng.sample(sorted(E), min(k, len(E))):
            E.discard((a, b)); E.add((b, a))
    elif type_err == 'E4':    # suppressions
        for e in rng.sample(sorted(E), min(k, len(E))):
            E.discard(e)
    elif type_err == 'E5':    # cycles : C -> A alors que A mène déjà à C
        G = nx.DiGraph(list(E))
        candidats = sorted(E); rng.shuffle(candidats)
        for a, b in candidats:
            if k == 0:
                break
            c = rng.choice(list(nx.descendants(G, b)) + [b])
            if c != a and (c, a) not in E:
                E.add((c, a)); G.add_edge(c, a); k -= 1
    return sorted(E)

def graphe(nom, graine=GRAINE, taux=0.0):
    if nom == 'expert':
        return A_expert
    if nom == 'aucun':
        return np.zeros((N, N), dtype=np.float32)
    if nom == 'aleatoire':
        rng = random.Random(graine)
        E = set()
        while len(E) < len(aretes_expert):
            a, b = rng.sample(range(N), 2)
            E.add((a, b))
        return vers_matrice(E)
    return vers_matrice(corrompre(aretes_expert, nom, taux, graine))

def f1_aretes(A, B=A_expert):
    P, V = set(zip(*np.nonzero(A))), set(zip(*np.nonzero(B)))
    tp = len(P & V); p = tp / max(1, len(P)); r = tp / max(1, len(V))
    return round(2 * p * r / (p + r), 3) if p + r else 0.0

print(f'{"graphe":8s} {"taux":>5s} {"flèches":>8s} {"F1/expert":>10s} {"acyclique":>10s}')
for t in ['E1', 'E2', 'E3', 'E4', 'E5']:
    for taux in [0.1, 0.3, 0.5]:
        A = graphe(t, 0, taux)
        print(f'{t:8s} {taux:5.0%} {int(A.sum()):8d} {f1_aretes(A):10.3f} {str(nx.is_directed_acyclic_graph(nx.DiGraph(A))):>10s}')

In [ ]:
# Cellule 6 — Entraînement et évaluation par sous-ensemble
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=128):
    modele.eval()
    Q, R, M, F = T
    ys, ps, fs = [], [], []
    for i in range(0, len(Q), batch):
        q, r, m, f = (x[i:i+batch].to(DEVICE) for x in (Q, R, M, F))
        p = torch.sigmoid(modele(q, r))
        c = m[:, 1:]
        ys.append(r[:, 1:][c].cpu().numpy()); ps.append(p[c].float().cpu().numpy()); fs.append(f[:, 1:][c].cpu().numpy())
    y, p, f = (np.concatenate(x) for x in (ys, ps, fs))
    auc = lambda s: round(float(roc_auc_score(y[s], p[s])), 4) if len(set(y[s])) == 2 else None
    tout = np.ones_like(f, dtype=bool)
    return {'AUC': auc(tout), 'AUC_1re': auc(f), 'AUC_rep': auc(~f), 'ACC': round(float(accuracy_score(y, p >= 0.5)), 4)}

def entrainer(nom_modele, A, fraction=1.0, graine=GRAINE, hp=HP):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = GraphKT(N, hp, A, indicateurs=(nom_modele == 'PrereqKT')).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    return {'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
            'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1)}
print('OK')

## Partie A — Suppressions E4 à 70 % et 90 %

In [ ]:
# Cellule A1 — E4 à 70 % et 90 % (PrereqKT, 1 % et 5 %, 5 graines), ajouté au fichier de l'étape 8
GRAINES = [0, 1, 2, 3, 4]
chemin_grille = os.path.join(RESULTATS, 'rq2_grille.csv')
grille_res = pd.read_csv(chemin_grille).to_dict('records') if os.path.exists(chemin_grille) else []
fait = {(x['modele'], x['graphe'], float(x['fraction']), float(x['taux']), int(x['graine'])) for x in grille_res}
for frac in [0.05, 0.01]:
    for taux in [0.7, 0.9]:
        for g_ in GRAINES:
            if ('PrereqKT', 'E4', frac, taux, g_) in fait:
                continue
            res = entrainer('PrereqKT', graphe('E4', graine=g_, taux=taux), fraction=frac, graine=g_)
            grille_res.append({'modele': 'PrereqKT', 'graphe': 'E4', 'taux': taux, 'fraction': frac, 'graine': g_, **res})
            pd.DataFrame(grille_res).to_csv(chemin_grille, index=False)
            print(f'E4 {taux:.0%} | {frac:.0%} | graine {g_} -> AUC 1res {res["AUC_1re_test"]}')
print('Partie A terminée.')

## Partie B — Graphe de Junyi généré par le LLM

**Paires candidates :** toutes les paires ordonnées (A, B) d'exercices du **même thème** (`topic`).
Interroger toutes les paires possibles (environ 700 000) serait trop long. La cellule B1 mesure **quelle part des flèches expertes** reste dans ce périmètre : c'est le rappel maximal atteignable, à mentionner dans l'article.

In [ ]:
# Cellule B1 — Paires candidates et couverture du graphe expert
# Périmètre : même thème (topic) par défaut ; même domaine (area) si le thème couvre moins de 70 % des flèches expertes
exos['texte'] = exos['name'].str.replace('_', ' ').str.strip()
def paires_par(colonne):
    groupes = exos.dropna(subset=[colonne]).groupby(colonne)['exo_id'].apply(list)
    return [(a, b) for ids in groupes for a in ids for b in ids if a != b]
couverture = {}
for colonne in ['topic', 'area']:
    c = paires_par(colonne)
    cs = set(c)
    couverture[colonne] = (c, sum(1 for e in aretes_expert if tuple(e) in cs) / len(aretes_expert))
    print(f'Périmètre « {colonne} » : {len(c):,} paires | flèches expertes couvertes : {couverture[colonne][1]:.1%}')
PERIMETRE = 'topic' if couverture['topic'][1] >= 0.70 or len(couverture['area'][0]) > 120_000 else 'area'
candidats, rappel_max = couverture[PERIMETRE]
print(f'\n=> Périmètre retenu : « {PERIMETRE} » — {len(candidats):,} paires, rappel maximal {rappel_max:.1%}')
print('Exemples :', [(exos.texte[a], exos.texte[b]) for a, b in candidats[:3]])

In [ ]:
# Cellule B2 — Chargement de Qwen2.5-7B (4 bits)
!pip install -q -U transformers accelerate bitsandbytes
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODELE_LLM = 'Qwen/Qwen2.5-7B-Instruct'      # si mémoire insuffisante : 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(MODELE_LLM)
tok.padding_side = 'left'
llm = AutoModelForCausalLM.from_pretrained(
    MODELE_LLM, device_map='auto',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))
llm.eval()
I_OUI = tok.encode('Yes', add_special_tokens=False)[0]
I_NON = tok.encode('No', add_special_tokens=False)[0]

SYSTEME = 'You are an expert mathematics teacher who designs curricula.'
GABARIT = ('Two exercises from an online mathematics course (topics: "{topic}" and "{topic_b}"):\n'
           'Exercise A: {a}\nExercise B: {b}\n'
           'Must a student master exercise A before attempting exercise B '
           '(is A a prerequisite of B)? Answer only Yes or No.')

@torch.no_grad()
def proba_oui(paires, taille_lot=32):
    textes = [tok.apply_chat_template(
                  [{'role': 'system', 'content': SYSTEME},
                   {'role': 'user', 'content': GABARIT.format(topic=str(exos.topic[a]).replace('-', ' '), topic_b=str(exos.topic[b]).replace('-', ' '),
                                                               a=exos.texte[a], b=exos.texte[b])}],
                  tokenize=False, add_generation_prompt=True) for a, b in paires]
    sortie = []
    for i in range(0, len(textes), taille_lot):
        lot = tok(textes[i:i + taille_lot], return_tensors='pt', padding=True).to(llm.device)
        z = llm(**lot).logits[:, -1, :].float()
        sortie += torch.softmax(torch.stack([z[:, I_OUI], z[:, I_NON]], 1), 1)[:, 0].cpu().tolist()
    return sortie

nv = dict(zip(exos.name, exos.exo_id))
if {'factoring_polynomials_1', 'solving_quadratics_by_factoring'} <= set(nv):
    print('Test (prérequis expert) :', proba_oui([(nv['factoring_polynomials_1'], nv['solving_quadratics_by_factoring'])]))
print('Modèle chargé :', MODELE_LLM)

In [ ]:
# Cellule B3 — Évaluation de toutes les paires candidates (sauvegarde régulière, reprise automatique)
chemin_scores = os.path.join(RESULTATS, 'junyi_llm_scores.csv')
scores = pd.read_csv(chemin_scores) if os.path.exists(chemin_scores) else pd.DataFrame(columns=['A', 'B', 'p_oui'])
deja = set(zip(scores.A.astype(int), scores.B.astype(int)))
reste = [p for p in candidats if p not in deja]
print(f'Déjà évaluées : {len(deja):,} | restantes : {len(reste):,}')
t0, BLOC = time.time(), 2048
for i in range(0, len(reste), BLOC):
    part = reste[i:i + BLOC]
    p = proba_oui(part)
    scores = pd.concat([scores, pd.DataFrame({'A': [a for a, _ in part], 'B': [b for _, b in part], 'p_oui': p})],
                       ignore_index=True)
    scores.to_csv(chemin_scores, index=False)
    fait_ = i + len(part)
    print(f'{fait_:,}/{len(reste):,} paires | {(time.time()-t0)/60:.1f} min | '
          f'reste ≈ {(time.time()-t0)/fait_*(len(reste)-fait_)/60:.0f} min')
print('Génération terminée :', len(scores), 'paires')

In [ ]:
# Cellule B4 — Graphes LLM : du plus prudent au plus généreux, et leur qualité face au graphe expert
import gc
if 'llm' in globals():
    del llm; gc.collect(); torch.cuda.empty_cache()      # libère le GPU pour les entraînements

scores = pd.read_csv(os.path.join(RESULTATS, 'junyi_llm_scores.csv'))
scores[['A', 'B']] = scores[['A', 'B']].astype(int)
scores = scores.sort_values('p_oui', ascending=False).reset_index(drop=True)
vrai = set(map(tuple, aretes_expert))
scores['expert'] = [(a, b) in vrai for a, b in zip(scores.A, scores.B)]
print(f"AUC du LLM sur les paires candidates : {roc_auc_score(scores.expert, scores.p_oui):.3f}")

E_star = len(aretes_expert)
GRAPHES_LLM = {}
for lib, k in [('LLM_x0.5', 0.5), ('LLM_x1', 1), ('LLM_x2', 2), ('LLM_x4', 4), ('LLM_x8', 8)]:
    GRAPHES_LLM[lib] = scores.head(int(k * E_star))                       # les k × |E*| paires les plus probables
GRAPHES_LLM['LLM_seuil0.5'] = scores[scores.p_oui >= 0.5]                  # décision « oui/non » brute du LLM

qualite = []
for lib, s in GRAPHES_LLM.items():
    E = list(zip(s.A, s.B))
    A = vers_matrice(E)
    tp = int(s.expert.sum())
    qualite.append({'graphe': lib, 'fleches': len(E), 'seuil_equivalent': round(float(s.p_oui.min()), 3) if len(s) else None,
                    'precision': round(tp / max(1, len(E)), 3), 'rappel': round(tp / E_star, 3),
                    'F1': f1_aretes(A), 'acyclique': nx.is_directed_acyclic_graph(nx.DiGraph(E)) if E else True})
qualite = pd.DataFrame(qualite)
qualite.to_csv(os.path.join(RESULTATS, 'junyi_llm_qualite.csv'), index=False)
print(qualite.to_string(index=False))

In [ ]:
# Cellule B5 — PrereqKT avec chaque graphe LLM (5 % puis 1 %, 5 graines ; reprise automatique)
GRAINES = [0, 1, 2, 3, 4]
chemin_llm = os.path.join(RESULTATS, 'rq2_graphes_llm.csv')
res_llm = pd.read_csv(chemin_llm).to_dict('records') if os.path.exists(chemin_llm) else []
fait = {(x['graphe'], float(x['fraction']), int(x['graine'])) for x in res_llm}
for frac in [0.05, 0.01]:
    for lib, s in GRAPHES_LLM.items():
        A = vers_matrice(list(zip(s.A, s.B)))
        for g_ in GRAINES:
            if (lib, frac, g_) in fait:
                continue
            res = entrainer('PrereqKT', A, fraction=frac, graine=g_)
            res_llm.append({'modele': 'PrereqKT', 'graphe': lib, 'fraction': frac, 'graine': g_, **res})
            pd.DataFrame(res_llm).to_csv(chemin_llm, index=False)
            print(f'{lib:13s} | {frac:.0%} | graine {g_} -> AUC 1res {res["AUC_1re_test"]}')

## Synthèse

**Figure clé :** en abscisse, le **rappel** du graphe (part des flèches expertes présentes) ; en ordonnée, le **bénéfice conservé**.
- La courbe **E4** (suppressions pures, précision = 1) montre ce que coûte le seul fait d'oublier des flèches.
- Les points **LLM** ont, en plus, de fausses flèches.
S'ils se placent près de la courbe E4, c'est surtout le **rappel** qui compte, et un graphe LLM généreux est préférable.

In [ ]:
# Cellule B6 — Résumé et figure à copier dans la conversation
import matplotlib.pyplot as plt
ref = pd.read_csv(os.path.join(RESULTATS, 'phase2_confirmation.csv'))
ref = ref[ref.modele == 'PrereqKT']
grille = pd.read_csv(os.path.join(RESULTATS, 'rq2_grille.csv'))
grille = grille[(grille.modele == 'PrereqKT') & (grille.graphe == 'E4')]
llm_res = pd.DataFrame(res_llm)
col = 'AUC_1re_test'

lignes = []
for frac in [0.05, 0.01]:
    base = ref[ref.fraction == frac].pivot_table(index='graine', columns='graphe', values=col)
    gain = base['expert'] - base['aucun']
    def conserve(serie):
        g = serie.index.intersection(gain.index)
        return 100 * ((serie.loc[g] - base.loc[g, 'aucun']) / gain.loc[g]).mean(), serie.loc[g].mean(), len(g)
    for taux, s in grille[grille.fraction == frac].groupby('taux'):
        c, m, n = conserve(s.set_index('graine')[col])
        lignes.append({'fraction': frac, 'graphe': f'E4 {taux:.0%}', 'type': 'E4', 'rappel': round(1 - taux, 2),
                       'precision': 1.0, 'n': n, 'AUC_1re': round(m, 4), 'benefice_conserve_%': round(c, 1)})
    for lib, s in llm_res[llm_res.fraction == frac].groupby('graphe'):
        q = qualite.set_index('graphe').loc[lib]
        c, m, n = conserve(s.set_index('graine')[col])
        lignes.append({'fraction': frac, 'graphe': lib, 'type': 'LLM', 'rappel': q.rappel, 'precision': q.precision,
                       'n': n, 'AUC_1re': round(m, 4), 'benefice_conserve_%': round(c, 1)})
    lignes.append({'fraction': frac, 'graphe': 'expert', 'type': 'ref', 'rappel': 1.0, 'precision': 1.0,
                   'n': len(base), 'AUC_1re': round(base['expert'].mean(), 4), 'benefice_conserve_%': 100.0})
    lignes.append({'fraction': frac, 'graphe': 'aleatoire', 'type': 'ref', 'rappel': None, 'precision': None,
                   'n': len(base), 'AUC_1re': round(base['aleatoire'].mean(), 4),
                   'benefice_conserve_%': round(100 * ((base['aleatoire'] - base['aucun']) / gain).mean(), 1)})
synth = pd.DataFrame(lignes)
synth.to_csv(os.path.join(RESULTATS, 'rq1_rq2_synthese_llm.csv'), index=False)

print(qualite.to_string(index=False))
for frac in [0.05, 0.01]:
    s = synth[synth.fraction == frac]
    print(f'\n===== PrereqKT | {frac:.0%} des élèves =====')
    print(s.drop(columns=['fraction']).to_string(index=False))

    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    e4 = s[s.type == 'E4'].sort_values('rappel')
    ax.plot(list(e4.rappel) + [1.0], list(e4['benefice_conserve_%']) + [100], 'o-', c='tab:red',
            label='E4 : suppressions seules (précision = 1)')
    l = s[s.type == 'LLM']
    ax.scatter(l.rappel, l['benefice_conserve_%'], s=60, c='tab:blue', zorder=3, label='graphes LLM')
    for _, r in l.iterrows():
        ax.annotate(f"{r.graphe.replace('LLM_', '')}\nP={r.precision:.2f}", (r.rappel, r['benefice_conserve_%']),
                    fontsize=7, xytext=(4, 4), textcoords='offset points')
    ax.axhline(s[s.graphe == 'aleatoire']['benefice_conserve_%'].iloc[0], ls='--', c='grey', lw=1, label='graphe aléatoire')
    ax.axhline(0, ls=':', c='black', lw=1, label='aucun graphe')
    ax.set_xlabel('Rappel du graphe (part des flèches expertes présentes)')
    ax.set_ylabel('Bénéfice conservé (%)')
    ax.set_title(f'PrereqKT — {frac:.0%} des élèves'); ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTATS, f'rq1_rq2_rappel_benefice_{int(frac*100)}pc.png'), dpi=200)
    plt.show()